# Yield Curve Construction & Term Structure Models

From-scratch implementations of yield curve bootstrapping, interpolation, Nelson-Siegel and Svensson models — built entirely with NumPy and SciPy.

## 1. Motivation

The **yield curve** (term structure of interest rates) is the most fundamental object in fixed-income markets. It determines:

- The fair price of every bond, swap, and interest rate derivative
- Forward rates that encode market expectations about future rates
- The discount factors used to value any stream of future cash flows

### Why the Yield Curve Matters

Imagine the yield curve as a "thermometer" for the economy. Its shape tells you what the collective wisdom of millions of market participants thinks about future growth, inflation, and monetary policy. When the curve inverts (short-term rates exceed long-term rates), it has historically been one of the most reliable recession predictors — correctly signaling 7 of the last 8 US recessions.

### What We Will Cover

Building a yield curve from market data requires several steps, each with its own challenges:

| Step | Technique | Challenge |
|------|-----------|-----------|
| 1. Extract spot rates | **Bootstrapping** | Only a few maturities are directly observable |
| 2. Fill in the gaps | **Interpolation** | Must avoid artifacts like negative forward rates |
| 3. Smooth the curve | **Parametric models** (Nelson-Siegel, Svensson) | Balance fit accuracy with smoothness |

We implement each technique from scratch.

> **CFA Exam Tip:** The CFA curriculum distinguishes between par curves, spot (zero) curves, and forward curves. You must understand how they relate and be able to derive one from another. The spot curve is the foundation — par rates and forward rates are both derived from it.### Why Every Finance Professional Needs to Understand the Yield Curve

The yield curve is used in virtually every area of finance:

| Application | How the yield curve is used |
|:---|:---|
| **Bond pricing** | Discount each cash flow at the appropriate spot rate |
| **Derivatives pricing** | BSM uses the risk-free rate; interest rate derivatives need the full curve |
| **Corporate finance** | WACC calculation requires the appropriate discount rate for each cash flow |
| **Economic forecasting** | Curve shape predicts recessions (inverted = recession likely) |
| **Central banking** | Monetary policy transmission — short rates affect long rates |
| **Risk management** | Duration, convexity, and scenario analysis all depend on the curve |

> **Key Concept:** A single "interest rate" doesn't exist. There is a different rate for every maturity — 3-month, 1-year, 5-year, 30-year. The yield curve plots all these rates as a function of maturity. Understanding how to construct, interpret, and model this curve is essential.


### The Three Curves at a Glance

Before we dive into construction techniques, it helps to understand the three types of yield curves you will encounter and why each exists.

**Par curve.** This is what you see quoted in the market. Each point on the par curve answers the question: "What coupon rate would make a bond of this maturity trade at exactly par (face value)?" For example, if the 5-year par rate is 5%, a 5-year bond paying a 5% coupon would be priced at \$100.

**Spot curve (zero curve).** The spot rate for maturity $T$ is the yield on a zero-coupon bond maturing at $T$. It represents the "pure" time value of money for that specific horizon — no reinvestment risk, no coupon effects. The spot curve is the theoretically correct discount curve, but zero-coupon bonds do not trade at every maturity, so we must extract spot rates from coupon bond prices.

**Forward curve.** Forward rates are the interest rates implied by today's spot curve for future periods. For example, the "1-year rate, 2 years from now" is derived from the 2-year and 3-year spot rates. Forward rates are the building blocks of interest rate derivatives (FRAs, swaps, caps, floors).

> **Key Concept:** These three curves contain exactly the same information, just expressed differently. If you know any one of them completely, you can derive the other two. The par curve is easiest to observe; the spot curve is most useful for pricing; the forward curve is most useful for speculation and hedging.

**A practical analogy.** Think of the spot curve as the "odometer reading" on a car trip — it tells you total distance covered at each point in time. The forward curve is like the "speedometer" — it tells you how fast you are going at each instant. And the par curve is like the "average speed" so far — a blended measure that smooths out variations.### Spot Rate vs Par Rate vs Forward Rate

| Curve | Also called | What it represents |
|:---|:---|:---|
| **Spot rate** $s(T)$ | Zero-coupon rate | Rate for a single payment at time $T$ |
| **Par rate** $c(T)$ | Yield to maturity of a par bond | Coupon rate that makes a $T$-year bond trade at par |
| **Forward rate** $f(t_1, t_2)$ | — | Rate agreed today for borrowing from $t_1$ to $t_2$ |

> **CFA Exam Tip:** Spot rates are the building blocks. Par rates and forward rates can both be derived from spot rates. Going the other direction: you can bootstrap spot rates from par rates (which are observable via bond prices). The forward rate between years 2 and 3 is: $(1+s_3)^3 = (1+s_2)^2(1+f_{2,3})$.

### A Simple Numerical Example

Suppose $s_1 = 3\%$, $s_2 = 3.5\%$, $s_3 = 4\%$.

**The 1-year forward rate starting in year 2:**
$$f_{1,2} = \frac{(1.035)^2}{(1.03)^1} - 1 = \frac{1.07122}{1.03} - 1 = 4.00\%$$

**The 1-year forward rate starting in year 3 (i.e. from year 2 to year 3):**
$$f_{2,3} = \frac{(1.04)^3}{(1.035)^2} - 1 = \frac{1.12486}{1.07122} - 1 = 5.01\%$$

> **Key Concept:** Forward rates are higher than spot rates when the curve is upward-sloping. This is because the forward rate must "compensate" for the lower earlier rates to make the overall compounded return match the longer spot rate.


## 2. SetupWe use SciPy for optimisation (fitting Nelson-Siegel/Svensson models) and interpolation. All bootstrapping and forward rate calculations are built from scratch.


In [ ]:
%matplotlib inline
import numpy as np
from scipy import linalg, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

## 3. Term Structure Theories

Before diving into the mechanics of curve construction, it is important to understand *why* the yield curve takes different shapes. Four classical theories offer explanations, each with different implications for investors and policymakers.

### Expectations Hypothesis

**The idea:** Long-term rates are simply the geometric average of expected future short-term rates:

$$[1 + r(0, T)]^T = \prod_{t=0}^{T-1} [1 + E_0(r(t, t+1))]$$

**What it means:** An upward-sloping curve implies the market expects rates to rise. A flat curve implies rates are expected to stay the same. An inverted curve implies rates are expected to fall.

**Limitation:** If this theory were strictly true, long-term bonds would offer no extra return over rolling short-term bonds. In practice, long-term bonds typically do offer a premium, which the next theory explains.

### Liquidity Preference Theory

**The idea:** Investors demand a **liquidity premium** for holding longer-term bonds because they are riskier (more sensitive to rate changes). Forward rates exceed expected future spot rates:

$$f(t, t+1) = E_0[r(t, t+1)] + L(t)$$

where $L(t) > 0$ and typically increases with maturity.

**What it means:** The yield curve is *biased upward* relative to pure rate expectations. Even if the market expects rates to stay flat, the curve will slope upward because of the liquidity premium. This explains why upward-sloping curves are the "normal" shape.

**Real-world example:** In 2024, with the Fed Funds rate around 5.25-5.50%, the 10-year Treasury yielded only about 4.5% — an inverted curve. Under the liquidity preference theory, this means the market expected *significant* rate cuts (enough to overcome the positive liquidity premium and still produce an inversion).

### Market Segmentation Theory

**The idea:** Different institutional investors have **preferred habitats** along the maturity spectrum. Supply and demand in each segment independently determine rates.

- **Money market funds** buy short-term instruments
- **Pension funds** buy long-term bonds to match their liabilities
- **Banks** buy medium-term bonds for asset-liability management

**What it means:** The yield curve's shape can be explained by the relative supply and demand at each maturity, not just expectations. Heavy pension fund buying at the long end can push long rates down, flattening the curve.

### Preferred Habitat Theory

A refinement of market segmentation: investors have preferred maturities but *will move to other segments if the yield premium is sufficient*. This combines elements of expectations and segmentation.

### Summary: What the Curve Shape Tells You

| Curve Shape | Expectations Hypothesis | Liquidity Preference | Segmentation |
|-------------|------------------------|---------------------|-------------|
| **Upward-sloping** | Rates expected to rise | Normal (liquidity premium) | More demand for short-term |
| **Flat** | Rates expected to stay same | Rates expected to fall slightly | Supply/demand balanced |
| **Inverted** | Rates expected to fall significantly | Rates expected to fall a lot | More demand for long-term |
| **Humped** | Near-term rise, then fall | Mixed | Mixed supply/demand |

> **Key Concept:** No single theory fully explains the yield curve. In practice, all four forces operate simultaneously. The expectations hypothesis provides the framework, liquidity preference adds a risk premium, and segmentation/habitat effects explain short-term distortions.

> **CFA Exam Tip:** The CFA exam often asks you to explain curve shapes using these theories. A favorite question: "Why might the curve be inverted?" Under expectations hypothesis: the market expects rate cuts. Under liquidity preference: the market expects *very large* rate cuts (enough to overcome the positive premium).### The Four Main Theories

Each theory offers a different explanation for why the yield curve has its shape:

#### 1. Pure Expectations Hypothesis
Forward rates equal expected future spot rates:
$$f(t_1, t_2) = E[s(t_2 - t_1) \text{ at time } t_1]$$

**Implication:** An upward-sloping curve means the market expects rates to RISE. A flat curve means rates are expected to stay the same.

**Problem:** This theory implies that long-term bonds have no risk premium. But long-term bonds are riskier (more interest rate sensitivity), so investors should demand compensation.

#### 2. Liquidity Preference Theory (Keynes/Hicks)
Forward rates = expected future spot rates + a **liquidity premium** $L(T)$ that increases with maturity:
$$f(t_1, t_2) = E[s(t_2 - t_1)] + L(t_2)$$

**Implication:** The curve is biased upward by the liquidity premium. An upward-sloping curve could mean rates are expected to rise OR that the liquidity premium is dominating. The curve must be significantly inverted before we can conclude that rates are expected to fall.

> **CFA Exam Tip:** Liquidity preference theory is the most widely accepted explanation among CFA candidates. It explains why the curve is usually upward-sloping — even when rate expectations are flat, the liquidity premium pushes long rates above short rates.

#### 3. Market Segmentation Theory
Different maturity segments of the bond market are dominated by different investor types who don't easily substitute across maturities:
- **Short end:** Banks, money market funds
- **Intermediate:** Insurance companies
- **Long end:** Pension funds

Supply and demand within each segment determines rates independently.

**Problem:** This theory can't explain why rates across different segments tend to move together.

#### 4. Preferred Habitat Theory (Modigliani/Sutch)
Investors prefer specific maturity ranges but WILL move to other maturities if compensated with a sufficient premium. This is a compromise between expectations and segmentation.

> **Key Concept:** In practice, most market participants use a combination of expectations + risk premium thinking. The yield curve reflects both market expectations about future rates AND compensation for bearing interest rate risk.


### Why Does the Curve Invert Before Recessions?

This is one of the most important questions in fixed income, and it ties all the theories together.

**The economic story.** When the economy is overheating, the central bank raises short-term rates aggressively to cool inflation. But bond investors, looking ahead, expect that the tightening will eventually slow the economy so much that the central bank will need to *cut* rates in the future. So:

- Short-term rates are high (the central bank has pushed them up)
- Long-term rates are lower (the market expects rates to fall)

This creates the inversion.

**Why it predicts recession.** The inversion itself does not *cause* recession. Rather, it reflects that the market collectively believes the central bank has overtightened. Historically, when the market reaches this judgment, it has usually been right. The 2-year vs 10-year spread inverted before recessions in 1980, 1981-82, 1990, 2001, 2008, and 2020.

> **Key Concept:** Yield curve inversion is a *signal*, not a cause. It tells you that the collective intelligence of the bond market believes the current monetary policy path leads to an economic downturn. Think of it as the bond market "voting" on the economy's future.

**A worked numerical example.** Suppose the current 1-year rate is 5.5% (the central bank has hiked rates). The market expects the 1-year rate to be 3.0% one year from now (expecting cuts). Under the pure expectations hypothesis:

$$[1 + r(0,2)]^2 = (1 + 0.055)(1 + 0.030) = 1.055 \times 1.030 = 1.08665$$

$$r(0,2) = 1.08665^{1/2} - 1 = 4.24\%$$

The 2-year rate (4.24%) is *below* the 1-year rate (5.5%) — the curve is inverted. The large expected rate cut (from 5.5% to 3.0%) is sufficient to pull the longer-term rate below the short-term rate.### The Three Signals from an Inverted Curve

1. **Rate cut expectations:** The market expects the central bank to cut short rates in the future (usually because of recession fears)
2. **Flight to safety:** During uncertainty, investors buy long-term government bonds (pushing long-term yields down)  
3. **Tightening monetary policy:** The central bank has raised short-term rates above the "neutral" rate

> **CFA Exam Tip:** The yield curve has inverted before EVERY U.S. recession since 1960, with only one false positive (1966). However, the lead time varies from 6 to 24 months, making precise timing impossible. The most watched spread is the **2-year vs 10-year Treasury** spread — when it goes negative, concern rises.


## 4. Bootstrapping Spot Rates

### The Problem

In the real world, we do not directly observe zero-coupon (spot) rates for all maturities. What we observe are prices of coupon-bearing bonds. The challenge is to extract the underlying spot rates from these observed prices.

### The Intuition

Bootstrapping works iteratively, starting from the shortest maturity and working outward:

1. The 6-month par rate directly gives us the 6-month spot rate (a 6-month par bond has only one cash flow).
2. For the 1-year par bond, we know the 6-month spot rate and can solve for the 1-year spot rate.
3. For the 1.5-year par bond, we know both shorter spot rates and can solve for the 1.5-year spot rate.
4. And so on...

### Step-by-Step Worked Example

Suppose we have three par bonds:

| Maturity | Par Rate | Price |
|:--------:|:--------:|:-----:|
| 0.5 yr | 4.0% | $100 |
| 1.0 yr | 4.2% | $100 |
| 1.5 yr | 4.4% | $100 |

All bonds have semi-annual coupons. We bootstrap the spot rates:

**Step 1 — 6-month spot rate ($z_{0.5}$):**

The 6-month par bond pays one coupon of $100 \times 0.04/2 = \$2$ plus the face value of $100 in 6 months:

$$100 = \frac{102}{(1 + z_{0.5}/2)^1} \implies z_{0.5} = 2 \times \left(\frac{102}{100} - 1\right) = 4.00\%$$

The first spot rate equals the first par rate (always true for the shortest maturity).

**Step 2 — 1-year spot rate ($z_{1.0}$):**

The 1-year par bond pays $2.10 at 6 months and $102.10 at 1 year:

$$100 = \frac{2.10}{(1 + 0.04/2)^1} + \frac{102.10}{(1 + z_{1.0}/2)^2}$$

$$100 = \frac{2.10}{1.02} + \frac{102.10}{(1 + z_{1.0}/2)^2}$$

$$100 = 2.0588 + \frac{102.10}{(1 + z_{1.0}/2)^2}$$

$$\frac{102.10}{97.9412} = (1 + z_{1.0}/2)^2 \implies z_{1.0} = 2 \times \left[(102.10/97.9412)^{1/2} - 1\right] \approx 4.2008\%$$

Notice the spot rate (4.2008%) is slightly higher than the par rate (4.2%). This is always the case when the curve is upward-sloping.

**Step 3 — 1.5-year spot rate ($z_{1.5}$):**

Same process — use the known $z_{0.5}$ and $z_{1.0}$ to discount the first two coupons, then solve for $z_{1.5}$.

### The General Formula

For a par bond with maturity $T_n$, coupon rate $c_n$, and price $P_n = 100$ (par):

$$P_n = \sum_{i=1}^{n-1} \frac{c_n / m}{(1 + z_i / m)^{i}} + \frac{c_n / m + 100}{(1 + z_n / m)^{n}}$$

Since we already know $z_1, \ldots, z_{n-1}$, we solve for $z_n$:

$$z_n = m \left[ \left( \frac{c_n/m + 100}{P_n - \sum_{i=1}^{n-1} \frac{c_n/m}{(1+z_i/m)^i}} \right)^{1/n} - 1 \right]$$

> **Key Concept:** The spot curve always lies *above* the par curve when the curve is upward-sloping, and *below* when it is downward-sloping. This is because the par rate is a weighted average of the spot rates, and if spot rates are increasing, the average is below the latest rate.

> **CFA Exam Tip:** Bootstrapping is a core exam topic. You should be able to perform the first 2-3 iterations by hand. The key insight is that each new spot rate depends on all previously computed spot rates.### Step-by-Step Bootstrapping with a Tiny Example

Let's bootstrap by hand before coding. Suppose we observe three par bonds:

| Maturity | Coupon rate | Price |
|:---:|:---:|:---:|
| 1 year | 4.0% | 100 |
| 2 year | 4.5% | 100 |
| 3 year | 5.0% | 100 |

**Step 1: 1-year spot rate**
The 1-year bond has one cash flow: $104 at year 1.
$$100 = \frac{104}{1 + s_1} \implies s_1 = 4.00\%$$

**Step 2: 2-year spot rate**
The 2-year bond pays $4.50 at year 1 and $104.50 at year 2:
$$100 = \frac{4.50}{1.04} + \frac{104.50}{(1 + s_2)^2}$$
$$100 = 4.3269 + \frac{104.50}{(1 + s_2)^2}$$
$$\frac{104.50}{(1 + s_2)^2} = 95.6731 \implies (1 + s_2)^2 = 1.0924 \implies s_2 = 4.512\%$$

**Step 3: 3-year spot rate**
$$100 = \frac{5.00}{1.04} + \frac{5.00}{(1.04512)^2} + \frac{105.00}{(1 + s_3)^3}$$
$$100 = 4.8077 + 4.5791 + \frac{105.00}{(1 + s_3)^3}$$
$$\frac{105.00}{(1 + s_3)^3} = 90.6132 \implies s_3 = 5.042\%$$

Notice that the spot rates (4.00%, 4.512%, 5.042%) are higher than the par rates (4.0%, 4.5%, 5.0%) — this is always the case when the par curve is upward-sloping.

> **CFA Exam Tip:** Bootstrapping is tested on the CFA exam. You must be able to:
> 1. Extract the 1-year spot from a 1-year par bond
> 2. Use that to extract the 2-year spot from a 2-year par bond
> 3. Continue iteratively for longer maturities


### Why the Spot Curve Diverges from the Par Curve

To build deeper intuition, consider what happens as you go further out along an upward-sloping curve.

A 10-year par bond pays coupons every six months for 10 years. Each coupon is discounted at the spot rate for its payment date. The par rate is essentially a "blended" rate that averages all 20 spot rates (weighted by cash flow size). Since early coupons are discounted at lower spot rates and later coupons at higher spot rates, the blend sits below the 10-year spot rate.

The divergence grows with maturity because there are more low-rate coupons pulling the average down. For a steeply upward-sloping curve, the 30-year spot rate can be significantly higher than the 30-year par rate.

> **CFA Exam Tip:** If asked "which curve lies above the other?", remember the ordering for an upward-sloping term structure: Forward > Spot > Par. For a downward-sloping (inverted) term structure, the ordering reverses: Par > Spot > Forward.> **Intuition:** For an upward-sloping par curve, early coupons are discounted at lower rates and later coupons at higher rates. But the YTM (par rate) is a single "average" rate applied to ALL cash flows. Since the bond's price is dominated by the large final payment (discounted at the highest spot rate), the "average" rate (YTM) must be pulled upward — but it can't reach the longest spot rate because the early coupons are still discounted at lower rates. The spot rate for the final maturity must therefore exceed the par rate.

> **Common Mistake:** Thinking that the par rate and spot rate are the same thing. They are only equal for the 1-year maturity (where there's only one cash flow). For longer maturities, they diverge — the spot rate is the "pure" rate for that maturity, while the par rate is a weighted average of all spot rates up to that maturity.


The code below bootstraps spot rates from a set of par bond prices and plots the resulting spot curve alongside the par curve.> **What to watch for:** The code implements the exact same recursion we worked through by hand above, but for more maturities. The resulting plot should show the spot curve lying above the par curve (for an upward-sloping curve). The gap between the two curves widens with maturity.


In [ ]:
def bootstrap_spot_rates(maturities, coupon_rates, prices, freq=2):
    """Bootstrap spot rates from coupon bond prices.
    
    Parameters
    ----------
    maturities : array — bond maturities in years
    coupon_rates : array — annual coupon rates
    prices : array — observed bond prices
    freq : int — coupon frequency
    
    Returns
    -------
    spot_maturities : array — maturity for each spot rate (every period)
    spot_rates : array — bootstrapped spot rates (annualized)
    """
    # Build complete grid of spot rates at every coupon period
    max_periods = int(max(maturities) * freq)
    spot_rates = np.zeros(max_periods)
    spot_maturities = np.arange(1, max_periods + 1) / freq
    
    # Map maturities to period indices
    mat_to_idx = {m: int(m * freq) - 1 for m in maturities}
    
    for k, (mat, cpn, price) in enumerate(zip(maturities, coupon_rates, prices)):
        n = int(mat * freq)  # number of periods
        c = 100.0 * cpn / freq  # coupon per period
        
        # Sum PV of known coupons
        pv_known = 0.0
        for i in range(n - 1):
            z_i = spot_rates[i]  # already bootstrapped
            pv_known += c / (1.0 + z_i / freq) ** (i + 1)
        
        # Solve for z_n
        remaining = price - pv_known
        final_cf = c + 100.0  # last coupon + face
        
        # (final_cf / remaining)^(1/n) - 1 = z_n / freq
        z_n = freq * ((final_cf / remaining) ** (1.0 / n) - 1.0)
        spot_rates[n - 1] = z_n
        
        # Interpolate for intermediate periods without direct observations
        if k > 0:
            prev_n = int(maturities[k-1] * freq)
            if n - prev_n > 1:
                # Linear interpolation between known spot rates
                for j in range(prev_n, n - 1):
                    alpha = (j + 1 - prev_n) / (n - prev_n)
                    spot_rates[j] = spot_rates[prev_n - 1] * (1 - alpha) + z_n * alpha
    
    return spot_maturities, spot_rates


# --- Market data: par bonds ---
maturities = np.array([0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0])
coupon_rates = np.array([0.040, 0.042, 0.044, 0.046, 0.050, 0.054, 0.057, 0.060])
prices = np.full_like(maturities, 100.0)  # par bonds

spot_mats, spot_rates = bootstrap_spot_rates(maturities, coupon_rates, prices)

print("Bootstrapped Spot Rates")
print("-" * 40)
print(f"{'Maturity':>10s} {'Par Rate':>10s} {'Spot Rate':>12s}")
print("-" * 40)
for mat, cpn in zip(maturities, coupon_rates):
    idx = int(mat * 2) - 1
    print(f"{mat:>10.1f} {cpn:>10.3%} {spot_rates[idx]:>12.4%}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(maturities, coupon_rates * 100, 'o-', color=PRIMARY, linewidth=2, markersize=8, label='Par rates')
ax.plot(spot_mats, spot_rates * 100, 's--', color=SECONDARY, linewidth=2, markersize=5, label='Spot rates')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Rate (%)')
ax.set_title('Bootstrapped Spot Rate Curve')
ax.legend()
plt.tight_layout()
plt.show()

**Interpreting the output:** The spot rate curve lies above the par rate curve at every maturity, as expected for an upward-sloping term structure. The gap between the two curves widens at longer maturities, because the compounding effect of higher spot rates at the long end increasingly exceeds the blended par rate.

### Why Spot Rates Exceed Par Rates (Upward-Sloping Case)

Think of the par rate as a "blended" rate — it is a weighted average of all spot rates up to that maturity. In an upward-sloping environment, the most recent (highest) spot rate is being averaged with lower earlier spot rates, so the blend is below the latest spot rate.> **CFA Exam Tip:** When the par curve is upward-sloping, the spot curve lies ABOVE it. When the par curve is downward-sloping, the spot curve lies BELOW it. This relationship follows from the mathematics of bootstrapping — an upward-sloping par curve means later cash flows are discounted at higher rates, which "pulls" the spot rate above the par rate.

> **Common Mistake:** Students often confuse YTM (par rate) with the spot rate. The YTM is a single rate that makes the bond's PV equal to its price — it's a complex average of all the spot rates applicable to the bond's cash flows. Different bonds with the same maturity but different coupons have different YTMs even if they're both priced fairly on the same spot curve.


## 5. Forward Rate Derivation

### What Is a Forward Rate?

A forward rate is the interest rate for a future period that is implied by today's spot rates. It answers the question: "If I wanted to lock in a borrowing rate for a loan that starts 2 years from now and ends 3 years from now, what rate could I guarantee today?"

### The Investment Horizon Argument

The key insight behind forward rates is the **no-arbitrage** condition. Consider two strategies for investing \$1 over two years:

**Strategy A: Buy a 2-year zero-coupon bond.**
Your ending wealth is $(1 + z_2)^2$, where $z_2$ is the 2-year spot rate.

**Strategy B: Buy a 1-year zero-coupon bond, then reinvest the proceeds for 1 more year at the forward rate.**
Your ending wealth is $(1 + z_1)(1 + f_{1,2})$, where $f_{1,2}$ is the forward rate for year 2.

Both strategies are known today (no future decisions required), so they must produce the same result to avoid arbitrage:

$$(1 + z_2)^2 = (1 + z_1)(1 + f_{1,2})$$

Solving for the forward rate:

$$f_{1,2} = \frac{(1 + z_2)^2}{(1 + z_1)} - 1$$

**Worked numerical example.** If $z_1 = 4.0\%$ and $z_2 = 4.5\%$:

$$f_{1,2} = \frac{(1.045)^2}{1.040} - 1 = \frac{1.092025}{1.040} - 1 = 5.002\%$$

The forward rate for year 2 is about 5.0%. Notice it is *higher* than the 2-year spot rate (4.5%). This makes intuitive sense: if the average of the first two years' rates is 4.5%, and the first year is only 4.0%, the second year must be above average to pull the mean up.

> **Key Concept:** Forward rates are "marginal" rates — the rate for the next period only. Spot rates are "average" rates — the average over all periods from now to maturity. Just like a student's semester GPA (forward rate) versus cumulative GPA (spot rate): if your cumulative GPA is rising, your latest semester GPA must be above the cumulative.

### The Formulas

Given spot rates $z(t_1)$ and $z(t_2)$, the forward rate for the period $[t_1, t_2]$ under continuous compounding:

$$f(t_1, t_2) = \frac{z(t_2) \cdot t_2 - z(t_1) \cdot t_1}{t_2 - t_1}$$

Under discrete compounding ($m$ times per year):

$$\left(1 + \frac{f}{m}\right)^{m(t_2 - t_1)} = \frac{(1 + z_2/m)^{m \cdot t_2}}{(1 + z_1/m)^{m \cdot t_1}}$$

> **CFA Exam Tip:** You must be able to compute forward rates from spot rates by hand. The exam will give you two or three spot rates and ask for the implied forward rate. Always set up the no-arbitrage equation: the two investment strategies must produce the same outcome.

The code below extracts one-period forward rates from the bootstrapped spot curve.

In [ ]:
def forward_rates_from_spots(spot_maturities, spot_rates, freq=2):
    """Extract forward rates from spot rate curve.
    
    Returns 1-period forward rates: f(t_i, t_{i+1}).
    """
    n = len(spot_maturities)
    fwd_rates = np.zeros(n)
    
    # First forward rate = first spot rate
    fwd_rates[0] = spot_rates[0]
    
    for i in range(1, n):
        t1 = spot_maturities[i - 1]
        t2 = spot_maturities[i]
        z1 = spot_rates[i - 1]
        z2 = spot_rates[i]
        
        # Discrete forward rate
        n1 = int(t1 * freq)
        n2 = int(t2 * freq)
        growth_ratio = (1 + z2 / freq) ** n2 / (1 + z1 / freq) ** n1
        fwd_rates[i] = freq * (growth_ratio ** (1.0 / (n2 - n1)) - 1.0)
    
    return fwd_rates


fwd_rates = forward_rates_from_spots(spot_mats, spot_rates)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(spot_mats, spot_rates * 100, 'o-', color=PRIMARY, linewidth=2, label='Spot rates')
ax.step(spot_mats, fwd_rates * 100, where='mid', color=SECONDARY, linewidth=2, label='Forward rates')
ax.plot(maturities, coupon_rates * 100, 's', color=TERTIARY, markersize=10, label='Par rates', zorder=5)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Rate (%)')
ax.set_title('Par, Spot, and Forward Rate Curves')
ax.legend()
plt.tight_layout()
plt.show()

**Interpreting the output:** The forward rate curve is the most volatile of the three curves, and it always lies above the spot curve when the spot curve is rising. This confirms the GPA analogy: if your cumulative GPA is rising, your latest semester GPA must be above the cumulative.

### The Three Curves Side by Side

| Curve | What It Represents | Relationship |
|-------|-------------------|-------------|
| **Par curve** | YTM of par-priced coupon bonds | Blended average of spot rates (weighted by cash flows) |
| **Spot curve** | Rate for zero-coupon investments at each maturity | Geometric average of forward rates |
| **Forward curve** | Implied rate for future single periods | Marginal rate — the "next semester GPA" |

For an upward-sloping curve: Forward > Spot > Par at every maturity.> **Key Concept:** Forward rates amplify information in the spot curve. Small changes in the spot curve produce large changes in forward rates. This is both useful (forward rates reveal market expectations more clearly) and dangerous (estimation errors in spot rates are amplified in forward rates).

> **Important:** Under the **expectations hypothesis**, forward rates equal expected future spot rates: $f_{1,2} = E[s_1(t=1)]$. If this holds, an upward-sloping forward curve means the market expects rates to RISE. In reality, a liquidity premium biases forward rates upward, so forward rates typically overpredict future spot rates.


### Using Forward Rates in Practice: Locking In a Future Rate

Forward rates are not just theoretical constructs — they are directly tradeable. A company that knows it needs to borrow \$10 million in two years can use a **forward rate agreement (FRA)** to lock in the forward rate today, eliminating interest rate uncertainty.

**Example.** Suppose the 2-year spot rate is 4.5% and the 3-year spot rate is 4.8%. The implied 1-year forward rate starting in year 2 is:

$$f_{2,3} = \frac{(1.048)^3}{(1.045)^2} - 1 = \frac{1.15065}{1.09203} - 1 = 5.37\%$$

The company can enter an FRA today to guarantee borrowing at 5.37% for year 3. If rates turn out to be higher (say 6%), the company benefits from the locked-in lower rate. If rates turn out to be lower (say 4%), the company pays more than the prevailing rate — but had certainty, which was the goal.

> **Key Concept:** Forward rates represent "break-even" rates. If the actual future spot rate equals the forward rate, both strategies (locking in vs. waiting) produce the same result. The forward rate is the rate at which you are indifferent between the two strategies.### Worked Example: Locking In a Forward Rate

Suppose $s_1 = 3\%$ and $s_2 = 4\%$. The implied forward rate from year 1 to year 2 is:
$$f_{1,2} = \frac{(1.04)^2}{1.03} - 1 = 5.01\%$$

**How to lock in 5.01% for year 2:**
1. Today: Borrow \$1 for 1 year at 3%. You owe \$1.03 at year 1.
2. Today: Invest \$1 for 2 years at 4%. You receive \$1.0816 at year 2.
3. At year 1: Pay back \$1.03 (using new borrowing if needed).
4. Net: You invested \$1.03 at year 1 and received \$1.0816 at year 2.

Return from year 1 to 2: $1.0816/1.03 - 1 = 5.01\%$ ✓

> **Key Concept:** This locking-in strategy is an **arbitrage argument**. If actual rates in year 2 turn out to be different from $f_{1,2}$, it doesn't matter — you've already locked in the rate. This is how the forward rate is determined by no-arbitrage, regardless of anyone's expectations.


## 6. Interpolation Methods

### The Problem

We only observe bond prices at a few specific maturities (0.5, 1, 2, 3, 5, 7, 10 years in our example). But we need spot rates at *every* maturity — for example, to price a 4.7-year bond. This requires interpolation.

### Why Interpolation Choice Matters

Not all interpolation methods are equal. The choice of method affects not just the spot curve but also the implied forward rates. A poorly chosen interpolation can produce:
- **Negative forward rates** (economically nonsensical in most environments)
- **Discontinuous forward rates** (jumps at observation points)
- **Oscillating curves** (wiggles between data points)

### Common Methods

| Method | Pros | Cons | Used By |
|--------|------|------|---------|
| **Linear** | Simple, fast | Discontinuous forward rates | Quick estimates |
| **Cubic spline** | Smooth spot curve, smooth forwards | Can produce negative forwards | Academic research |
| **Monotone convex (PCHIP)** | Monotonicity-preserving, positive forwards | Slightly less smooth | Production systems |

> **Key Concept:** The interpolation method matters more for forward rates than for spot rates. Even a crude interpolation gives reasonable spot rates, but the implied forward rates can be wildly different and economically nonsensical.

### The Mathematical Reason Forward Rates Are More Sensitive

Recall that the instantaneous forward rate is the derivative of the function $r(t) \cdot t$:

$$f(t) = \frac{d}{dt}[r(t) \cdot t] = r(t) + t \cdot r'(t)$$

So the forward rate depends on the *slope* of the spot curve, not just its level. Differentiation amplifies any kinks or wiggles in the spot curve. A barely visible bump in the spot curve can produce wild swings in the forward rate. This is why smooth interpolation methods are so important in practice.### Why Interpolation Matters

In practice, you might need the 3.7-year spot rate, but you only have bonds at 3-year and 5-year maturities. Interpolation fills in the gaps.

**The key trade-off:**
- **Too simple (linear):** Produces kinks in the spot curve and discontinuous forward rates
- **Too flexible (cubic spline):** Can oscillate and produce negative forward rates  
- **Just right (monotone convex / PCHIP):** Smooth, shape-preserving, non-negative forwards

> **Important:** The forward rate is the derivative of the discount function. If the spot curve has kinks (from linear interpolation), the forward rate has jumps. If the spot curve oscillates (from unconstrained cubic splines), the forward rate can go negative. This is why the choice of interpolation method has practical consequences for derivative pricing.


The code compares all three interpolation methods on our bootstrapped spot curve. Pay special attention to the right panel showing implied forward rates — this is where the differences really matter.

In [ ]:
from scipy.interpolate import CubicSpline, PchipInterpolator

def linear_interp(x_known, y_known, x_new):
    """Piecewise linear interpolation from scratch."""
    y_new = np.zeros_like(x_new)
    for k, x in enumerate(x_new):
        if x <= x_known[0]:
            y_new[k] = y_known[0]
        elif x >= x_known[-1]:
            y_new[k] = y_known[-1]
        else:
            i = np.searchsorted(x_known, x) - 1
            t = (x - x_known[i]) / (x_known[i+1] - x_known[i])
            y_new[k] = y_known[i] * (1 - t) + y_known[i+1] * t
    return y_new


def cubic_spline_interp(x_known, y_known, x_new):
    """Natural cubic spline interpolation using scipy."""
    cs = CubicSpline(x_known, y_known, bc_type='natural')
    return cs(x_new)


def monotone_convex_interp(x_known, y_known, x_new):
    """Monotone convex interpolation (Hagan-West style) using PCHIP."""
    pchip = PchipInterpolator(x_known, y_known)
    return pchip(x_new)


# --- Compare interpolation methods ---
# Use bootstrapped spot rates at observed maturities
obs_mats = maturities
obs_spots = np.array([spot_rates[int(m * 2) - 1] for m in obs_mats])

fine_grid = np.linspace(0.5, 10.0, 200)

lin_interp = linear_interp(obs_mats, obs_spots, fine_grid)
cs_interp = cubic_spline_interp(obs_mats, obs_spots, fine_grid)
mc_interp = monotone_convex_interp(obs_mats, obs_spots, fine_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: interpolated spot curves
ax = axes[0]
ax.plot(fine_grid, lin_interp * 100, color=PRIMARY, linewidth=2, label='Linear')
ax.plot(fine_grid, cs_interp * 100, color=SECONDARY, linewidth=2, label='Cubic spline')
ax.plot(fine_grid, mc_interp * 100, color=TERTIARY, linewidth=2, label='Monotone (PCHIP)')
ax.plot(obs_mats, obs_spots * 100, 'ko', markersize=8, label='Observed', zorder=5)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Spot Rate (%)')
ax.set_title('Interpolated Spot Curves')
ax.legend()

# Right: implied forward rates (numerical derivative)
ax = axes[1]
dt = fine_grid[1] - fine_grid[0]
for interp_vals, label, color in [(lin_interp, 'Linear', PRIMARY),
                                   (cs_interp, 'Cubic spline', SECONDARY),
                                   (mc_interp, 'Monotone', TERTIARY)]:
    # f(t) = d/dt [r(t)*t] = r(t) + t*r'(t)
    rt = interp_vals * fine_grid
    fwd = np.gradient(rt, fine_grid)
    ax.plot(fine_grid, fwd * 100, color=color, linewidth=2, label=label)

ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Instantaneous Forward Rate (%)')
ax.set_title('Implied Forward Rates from Interpolation')
ax.legend()

plt.tight_layout()
plt.show()

**Interpreting the output:**

- **Left panel (spot curves):** All three methods produce similar-looking spot curves. You might think the choice does not matter — but look at the right panel.
- **Right panel (forward rates):** The differences are dramatic. Linear interpolation produces sharp jumps at observation points. Cubic spline is smooth but can overshoot. Monotone (PCHIP) provides a good balance of smoothness and stability.

In production systems, the monotone convex method (or similar shape-preserving interpolators) is preferred because it avoids the economically unreasonable forward rate artifacts that plague simpler methods.

> **CFA Exam Tip:** You are unlikely to be asked to implement interpolation on the exam, but you should understand the trade-off: simpler methods can produce unreasonable forward rates, while more sophisticated methods preserve economic reasonableness. The key principle is that forward rates should be positive and smooth.> **Key Concept:** The choice of interpolation method matters most for **forward rates** (right panel). Cubic spline interpolation can produce negative forward rates (which may be unrealistic), while monotone convex (PCHIP) preserves shape and avoids oscillation. For spot rates (left panel), the differences are small.

> **Common Mistake:** Linear interpolation is simple but produces discontinuous forward rates (kinks). This causes problems for derivative pricing because forward rates should be smooth. In practice, most trading desks use monotone convex or tension spline interpolation.


## 7. Nelson-Siegel Model

### The Idea: A Parsimonious Parametric Curve

Rather than interpolating between points, we can fit a smooth parametric function to the entire yield curve. The **Nelson-Siegel (1987)** model is the most widely used, with just four parameters:

$$r(\tau) = \beta_0 + \beta_1 \left(\frac{1 - e^{-\lambda \tau}}{\lambda \tau}\right) + \beta_2 \left(\frac{1 - e^{-\lambda \tau}}{\lambda \tau} - e^{-\lambda \tau}\right)$$

### What Each Parameter Controls

| Parameter | Name | Effect on Curve | Interpretation |
|-----------|------|-----------------|----------------|
| $\beta_0$ | Level | Shifts entire curve up/down | Long-term interest rate (as $\tau \to \infty$, $r \to \beta_0$) |
| $\beta_1$ | Slope | Tilts the curve | Short-term deviation from long-term level. Negative = upward-sloping |
| $\beta_2$ | Curvature | Creates a hump or trough | Medium-term bulge. Negative = hump, positive = trough |
| $\lambda$ | Decay | Controls hump location | Larger $\lambda$ = hump occurs at shorter maturities |

### Intuition for Each Component

Think of the Nelson-Siegel model as building the yield curve from three "building blocks":

1. **Level ($\beta_0$):** A flat line at the long-term rate. This is the "base" of the curve.
2. **Slope ($\beta_1$):** An exponentially decaying function that starts at $\beta_1$ for short maturities and fades to 0 at long maturities. This creates the tilt.
3. **Curvature ($\beta_2$):** A hump-shaped function that peaks at medium maturities and is zero at both the very short and very long ends. This creates the belly.

At maturity $\tau = 0$: $r(0) = \beta_0 + \beta_1$ (the short rate is level + slope).

At maturity $\tau \to \infty$: $r(\infty) = \beta_0$ (only the level persists).

### Why Central Banks Love Nelson-Siegel

Central banks (including the ECB, Fed, and BIS) routinely use Nelson-Siegel and Svensson models to summarize yield curve data. The parameters are economically interpretable: you can track how the "level" (long-term rates), "slope" (curve steepness), and "curvature" (belly) evolve over time as monetary policy changes.

> **Key Concept:** The Nelson-Siegel model is a *dimension reduction* tool. Instead of storing spot rates at hundreds of maturities, you store just four numbers ($\beta_0, \beta_1, \beta_2, \lambda$) that reconstruct the entire curve. This is why it is used in databases, risk systems, and regulatory reporting.

> **CFA Exam Tip:** You are not expected to fit Nelson-Siegel models on the exam, but you should know that $\beta_0$ represents the long-run level, $\beta_1$ represents the slope (steepness), and $\beta_2$ represents the curvature (hump). Questions may ask you to interpret fitted parameters.### The Nelson-Siegel Formula Decomposed

$$y(\tau) = \underbrace{\beta_0}_{\text{level}} + \underbrace{\beta_1 \left[\frac{1 - e^{-\lambda\tau}}{\lambda\tau}\right]}_{\text{slope}} + \underbrace{\beta_2 \left[\frac{1 - e^{-\lambda\tau}}{\lambda\tau} - e^{-\lambda\tau}\right]}_{\text{curvature}}$$

Each component has a clear shape:

| Component | At $\tau = 0$ | At $\tau \to \infty$ | Shape |
|:---|:---:|:---:|:---|
| Level ($\beta_0$) | $\beta_0$ | $\beta_0$ | Horizontal line |
| Slope ($\beta_1 \times \text{factor}$) | $\beta_1$ | 0 | Decays from $\beta_1$ to 0 |
| Curvature ($\beta_2 \times \text{factor}$) | 0 | 0 | Hump (peaks at intermediate $\tau$) |

**Reading the parameters:**
- $\beta_0 + \beta_1$ = short rate (instantaneous maturity)
- $\beta_0$ = long rate (infinite maturity)
- $\beta_1 < 0$ → upward-sloping curve; $\beta_1 > 0$ → downward-sloping
- $|\beta_2|$ controls the size of the hump/trough
- $\lambda$ controls WHERE the hump occurs (larger $\lambda$ → hump at shorter maturities)


### A Worked Example: Reading Nelson-Siegel Parameters

Suppose a central bank report gives the following fitted Nelson-Siegel parameters:

| Parameter | Value |
|-----------|-------|
| $\beta_0$ | 6.0% |
| $\beta_1$ | -2.0% |
| $\beta_2$ | -1.5% |
| $\lambda$ | 0.5 |

**Reading the parameters:**

- **Long-term rate** $= \beta_0 = 6.0\%$. The market expects rates to settle around 6% in the long run.
- **Short-term rate** $= \beta_0 + \beta_1 = 6.0\% + (-2.0\%) = 4.0\%$. Short rates are currently 4%.
- **Slope** $= \beta_1 = -2.0\%$. Since $\beta_1$ is negative, the curve slopes upward (short rates below long rates). The "spread" from short to long is about 2 percentage points.
- **Curvature** $= \beta_2 = -1.5\%$. Negative $\beta_2$ creates a hump in the middle of the curve. Medium-term rates are pushed up relative to a straight line between short and long rates.

If the central bank raises the policy rate (short-term rate), we would expect $\beta_1$ to become less negative (or positive), flattening or inverting the curve.

> **Key Concept:** Tracking Nelson-Siegel parameters over time gives you a compact summary of how the yield curve is evolving. A rising $\beta_0$ means long-term rates are going up (perhaps due to inflation expectations). A $\beta_1$ moving from negative to positive means the curve is flattening toward inversion.> **Common Mistake:** Students sometimes try to interpret Nelson-Siegel parameters individually, but they interact. A change in $\lambda$ shifts where the curvature peaks, which changes the effective contribution of $\beta_2$. Always look at the resulting CURVE shape, not just individual parameter values.

### Quick Parameter Estimation Heuristics

If you need to set initial parameter guesses for optimisation:
- $\beta_0 \approx$ the longest observed yield (10+ years)
- $\beta_1 \approx$ (short rate $-$ long rate), typically negative for normal curves
- $\beta_2 \approx$ 2 × (mid-range yield $-$ average of short and long yields)
- $\lambda \approx$ 1 / (maturity of maximum curvature in years)


The code below fits the Nelson-Siegel model to our bootstrapped spot rates and visualizes the fit.
> **What to watch for:** The fitted curve should pass close to the bootstrapped data points. The residuals (differences) should be small and randomly scattered — any systematic pattern suggests the model is missing a feature of the true curve shape.


In [ ]:
def nelson_siegel(tau, beta0, beta1, beta2, lam):
    """Nelson-Siegel yield curve model.
    
    Parameters
    ----------
    tau : array — maturities
    beta0, beta1, beta2 : float — level, slope, curvature
    lam : float — decay parameter
    """
    tau = np.asarray(tau, dtype=float)
    x = lam * tau
    # Avoid division by zero
    with np.errstate(invalid='ignore', divide='ignore'):
        factor1 = np.where(x < 1e-10, 1.0, (1.0 - np.exp(-x)) / x)
        factor2 = factor1 - np.exp(-x)
    
    return beta0 + beta1 * factor1 + beta2 * factor2


def fit_nelson_siegel(maturities, rates):
    """Fit Nelson-Siegel model via least squares."""
    def objective(params):
        beta0, beta1, beta2, lam = params
        if lam <= 0:
            return 1e10
        model_rates = nelson_siegel(maturities, beta0, beta1, beta2, lam)
        return np.sum((model_rates - rates) ** 2)
    
    # Initial guess
    x0 = [rates[-1], rates[0] - rates[-1], 0.0, 0.5]
    result = optimize.minimize(objective, x0, method='Nelder-Mead',
                               options={'maxiter': 10000, 'xatol': 1e-10, 'fatol': 1e-10})
    return result.x


# --- Fit to bootstrapped spot rates ---
params_ns = fit_nelson_siegel(obs_mats, obs_spots)
beta0, beta1, beta2, lam = params_ns

print("Nelson-Siegel Fitted Parameters")
print(f"  β₀ (level):     {beta0:.6f}")
print(f"  β₁ (slope):     {beta1:.6f}")
print(f"  β₂ (curvature): {beta2:.6f}")
print(f"  λ (decay):      {lam:.6f}")

# Evaluate on fine grid
ns_curve = nelson_siegel(fine_grid, *params_ns)

# Fitting error
ns_fitted = nelson_siegel(obs_mats, *params_ns)
rmse = np.sqrt(np.mean((ns_fitted - obs_spots) ** 2))
print(f"\n  RMSE: {rmse * 10000:.2f} bps")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fine_grid, ns_curve * 100, color=PRIMARY, linewidth=2, label='Nelson-Siegel fit')
ax.plot(obs_mats, obs_spots * 100, 'ko', markersize=8, label='Observed spot rates', zorder=5)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Rate (%)')
ax.set_title('Nelson-Siegel Model Fit')
ax.legend()
plt.tight_layout()
plt.show()

**Interpreting the output:** The fitted parameters tell a clear story:
- $\beta_0$ (level) is near the long-end spot rate — the long-term equilibrium rate.
- $\beta_1$ (slope) is negative — the curve slopes upward from short to long maturities.
- $\beta_2$ (curvature) captures any hump or dip in the middle of the curve.
- $\lambda$ (decay) determines where the slope and curvature effects peak.

The RMSE in basis points tells us how well the smooth parametric curve fits the actual bootstrapped spot rates. A few basis points of error is excellent for most applications.

> **Key Concept:** A low RMSE means the model captures the shape of the curve well. But be cautious: a model with *zero* error would be overfitting. Nelson-Siegel's strength is that it produces a smooth, economically plausible curve even if it does not pass exactly through every data point.> **Key Concept:** The beauty of Nelson-Siegel is that each parameter has a clear economic interpretation. When a central bank publishes NS parameters, you can immediately understand the curve:
> - Increasing $\beta_0$ → parallel shift up (all rates rise)
> - Decreasing $\beta_1$ (more negative) → steepening (short rates fall or long rates rise)
> - Increasing $|\beta_2|$ → more curvature (bigger hump or trough)
> - Increasing $\lambda$ → the hump shifts toward shorter maturities

This interpretability is why Nelson-Siegel is used by over 20 central banks worldwide for yield curve modelling.


## 8. Svensson Extension

The **Svensson (1994)** model adds a second curvature term to capture more complex shapes, including double humps:

$$r(\tau) = \beta_0 + \beta_1 \frac{1-e^{-\lambda_1 \tau}}{\lambda_1 \tau} + \beta_2 \left(\frac{1-e^{-\lambda_1 \tau}}{\lambda_1 \tau} - e^{-\lambda_1 \tau}\right) + \beta_3 \left(\frac{1-e^{-\lambda_2 \tau}}{\lambda_2 \tau} - e^{-\lambda_2 \tau}\right)$$

The extra two parameters ($\beta_3$ and $\lambda_2$) provide additional flexibility. The ECB officially uses the Svensson model for its yield curve estimates.

### Nelson-Siegel vs Svensson: When to Use Which

| Feature | Nelson-Siegel (4 params) | Svensson (6 params) |
|---------|------------------------|---------------------|
| Curve shapes | Normal, flat, inverted, single hump | All of above + double hump |
| Overfitting risk | Lower | Higher (more parameters) |
| Robustness | More stable optimization | Can be harder to fit |
| Used by | BIS, many central banks | ECB, Bundesbank |

> **Common Mistake:** Adding more parameters always improves in-sample fit, but can reduce out-of-sample stability. Svensson sometimes produces unreliable extrapolation beyond the observed maturity range.

### When Does Svensson Make a Difference?

For a simple upward-sloping curve, Nelson-Siegel is usually sufficient. Svensson becomes valuable when the curve has an unusual shape that Nelson-Siegel cannot capture with just one curvature term. Common scenarios include:

1. **Double hump curves.** Sometimes the yield curve rises, dips in the 2-3 year range (due to specific monetary policy expectations), then rises again. Nelson-Siegel can only produce a single hump.

2. **Kinked curves.** During financial stress, the short end may be distorted by central bank intervention while the long end reflects market expectations. The extra curvature term helps capture this disconnect.

3. **Very long maturities.** When fitting curves out to 30 or 50 years, the extra flexibility of Svensson helps the long end behave sensibly while still fitting the shorter maturities well.

> **CFA Exam Tip:** You do not need to know the Svensson formula for the exam. The key exam-relevant insight is that parametric models trade off *parsimony* (fewer parameters = more stable) against *flexibility* (more parameters = better fit to complex shapes). This trade-off applies throughout quantitative finance, not just to yield curves.> **Important:** Central banks that publish yield curve parameters often use Svensson rather than Nelson-Siegel, because government bond markets can exhibit complex shapes (especially in countries with specific supply concentrations at certain maturities). The ECB, Bundesbank, and Bank of England all use Svensson.

**Practical calibration tip:** When fitting Svensson, the extra parameters can cause the optimiser to find local minima. Start from multiple initial guesses and pick the fit with the lowest RMSE. Constrain $\lambda_1, \lambda_2 > 0$ and $\lambda_1 \neq \lambda_2$ to ensure identifiability.


The code below fits the Svensson model and compares it to the Nelson-Siegel fit.
> **What to watch for:** Compare the RMSE of Svensson vs Nelson-Siegel. For a simple curve shape, the improvement may be minimal; for curves with two humps, Svensson should fit noticeably better.


In [ ]:
def svensson(tau, beta0, beta1, beta2, beta3, lam1, lam2):
    """Svensson yield curve model."""
    tau = np.asarray(tau, dtype=float)
    x1 = lam1 * tau
    x2 = lam2 * tau
    
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(x1 < 1e-10, 1.0, (1.0 - np.exp(-x1)) / x1)
        f2 = f1 - np.exp(-x1)
        f3_part = np.where(x2 < 1e-10, 1.0, (1.0 - np.exp(-x2)) / x2)
        f3 = f3_part - np.exp(-x2)
    
    return beta0 + beta1 * f1 + beta2 * f2 + beta3 * f3


def fit_svensson(maturities, rates):
    """Fit Svensson model via least squares."""
    def objective(params):
        beta0, beta1, beta2, beta3, lam1, lam2 = params
        if lam1 <= 0 or lam2 <= 0:
            return 1e10
        model_rates = svensson(maturities, *params)
        return np.sum((model_rates - rates) ** 2)
    
    # Initial guess from Nelson-Siegel
    x0 = [params_ns[0], params_ns[1], params_ns[2], 0.0, params_ns[3], 1.0]
    result = optimize.minimize(objective, x0, method='Nelder-Mead',
                               options={'maxiter': 50000, 'xatol': 1e-12, 'fatol': 1e-12})
    return result.x


params_sv = fit_svensson(obs_mats, obs_spots)

print("Svensson Fitted Parameters")
labels = ['β₀', 'β₁', 'β₂', 'β₃', 'λ₁', 'λ₂']
for label, val in zip(labels, params_sv):
    print(f"  {label}: {val:.6f}")

sv_curve = svensson(fine_grid, *params_sv)
sv_fitted = svensson(obs_mats, *params_sv)
rmse_sv = np.sqrt(np.mean((sv_fitted - obs_spots) ** 2))
print(f"\n  RMSE: {rmse_sv * 10000:.2f} bps")

# Compare
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fine_grid, ns_curve * 100, color=PRIMARY, linewidth=2, label=f'Nelson-Siegel (RMSE: {rmse*10000:.2f} bps)')
ax.plot(fine_grid, sv_curve * 100, '--', color=SECONDARY, linewidth=2, label=f'Svensson (RMSE: {rmse_sv*10000:.2f} bps)')
ax.plot(obs_mats, obs_spots * 100, 'ko', markersize=8, label='Observed', zorder=5)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Rate (%)')
ax.set_title('Nelson-Siegel vs Svensson')
ax.legend()
plt.tight_layout()
plt.show()

**Interpreting the output:** The Svensson model achieves a lower RMSE (better fit) because it has two more parameters. For our simple upward-sloping curve, the improvement is modest — Nelson-Siegel already fits well. The Svensson extension becomes more valuable for curves with unusual shapes (e.g., a dip in the 2-3 year range followed by a rise).

> **Key Concept:** In model selection, the principle of parsimony (Occam's Razor) tells us to prefer the simpler model unless the extra complexity provides a meaningful improvement. For typical yield curve shapes, Nelson-Siegel's 4 parameters are sufficient. Use Svensson's 6 parameters only when the curve has features that Nelson-Siegel genuinely cannot capture.> **Important:** More parameters don't always mean a better model. With 6 parameters, Svensson can overfit — especially if the data has few maturities. The principle of parsimony suggests: **use Nelson-Siegel (4 parameters) unless you have clear evidence of two humps in the curve.** If the improvement in RMSE from adding the extra two parameters is small relative to your data uncertainty, stick with NS.

> **CFA Exam Tip:** You don't need to memorise the Svensson formula for the CFA exam, but you should know that parametric models like Nelson-Siegel use a small number of parameters to describe the entire yield curve, making them useful for smoothing, interpolation, and scenario generation.


## 9. Yield Curve Shapes and Economic Interpretation

The yield curve can take several characteristic shapes, each carrying important economic signals:

### Normal (Upward-Sloping)
- **Shape:** Short-term rates are lower than long-term rates
- **Economic signal:** The economy is expected to grow. Investors demand higher rates for locking up money longer. The central bank's policy rate is at or below the neutral rate.
- **Historical context:** This is the most common shape, observed roughly 70% of the time in the US.

### Inverted (Downward-Sloping)
- **Shape:** Short-term rates exceed long-term rates
- **Economic signal:** The market expects a recession and future rate cuts. The central bank has raised the policy rate above neutral to cool inflation.
- **Historical context:** Has preceded 7 of the last 8 US recessions (the one "false positive" was a brief inversion in 1998 during the LTCM crisis). The 2022-2024 inversion was the longest and deepest in 40+ years.

### Humped
- **Shape:** Rates rise for short-to-medium maturities, then decline for long maturities
- **Economic signal:** Transition period. The market may expect near-term rate hikes followed by eventual cuts. Often seen at the end of a tightening cycle.

### Flat
- **Shape:** Rates roughly equal across all maturities
- **Economic signal:** Maximum uncertainty. Often a transition between normal and inverted curves.

| Shape | Characteristics | Economic Signal |
|-------|----------------|----------------|
| **Normal** | Upward sloping | Expected growth, positive term premium |
| **Inverted** | Downward sloping | Recession signal, expected rate cuts |
| **Humped** | Rise then fall | Transition period, mixed expectations |
| **Flat** | Roughly constant | Uncertainty, potential turning point |

> **CFA Exam Tip:** The exam loves to test your understanding of yield curve shapes. Be prepared to explain what each shape implies under different term structure theories. An inverted curve under pure expectations means the market expects falling rates. Under liquidity preference, it means even stronger expectations of rate declines (strong enough to overcome the positive liquidity premium).### The Four Shapes and Their Economic Stories

| Shape | Short rates vs long rates | Economic interpretation |
|:---|:---|:---|
| **Normal (upward)** | Short < Long | Growth expected; investors demand term premium for longer maturities |
| **Inverted (downward)** | Short > Long | Recession feared; market expects rate cuts; flight to long bonds |
| **Humped** | Middle > Both ends | Transition period; near-term uncertainty but long-run stability expected |
| **Flat** | Roughly equal across maturities | Uncertainty about economic direction; often occurs during transitions |

> **Key Concept:** The yield curve shape is driven by the interaction of three forces:
> 1. **Rate expectations** (expectations hypothesis component)
> 2. **Risk premiums** (liquidity/term premium — usually positive and increasing with maturity)
> 3. **Supply and demand** in specific maturity segments (segmentation effects)


### Real-World Case Study: The 2022-2024 Inversion

The US yield curve inverted in mid-2022 and remained inverted for over two years — one of the longest inversions on record. Here is the story:

1. **The setup (2021-2022):** Post-pandemic inflation surged to 9%. The Fed began hiking aggressively, raising the Fed Funds rate from 0% to 5.25% in about 18 months.

2. **The inversion (July 2022):** The 2-year yield rose above the 10-year yield. Short-term rates reflected the Fed's aggressive stance; long-term rates stayed lower because the market expected the hiking cycle would eventually slow the economy and force rate cuts.

3. **The steepening debate (2023-2024):** As the inversion persisted, a key question emerged: would the curve normalize via a "bull steepener" (long rates stay put, short rates fall as the Fed cuts) or a "bear steepener" (short rates stay put, long rates rise on inflation fears)? The answer has profound implications for bond portfolio positioning.

This episode illustrates every theory we discussed: expectations (rate cuts expected), liquidity preference (the inversion required overcoming the term premium), and segmentation (massive Treasury issuance affected supply/demand at different maturities).

> **Key Concept:** Yield curve analysis is not just academic — it is a practical tool used by every fixed-income investor, central banker, and corporate treasurer to make decisions about borrowing, lending, and hedging.This episode illustrates the power and limitations of the yield curve as an economic indicator:
- **Power:** The inversion correctly signalled economic stress — the Fed's aggressive rate hikes created significant tightening
- **Limitation:** The curve was inverted for nearly 2 years before any recession materialised, making timing difficult
- **Context matters:** The curve inverted because the Fed raised short-term rates aggressively while long-term rates (reflecting long-run growth expectations) rose more slowly

> **Key Concept:** An inverted yield curve doesn't CAUSE a recession — it reflects market expectations that the central bank will eventually cut rates (which typically happens during recessions). The mechanism: high short rates → tighter credit → slower growth → Fed cuts → curve normalises.


The code below generates all four characteristic yield curve shapes using the Nelson-Siegel model with different parameter combinations.

In [ ]:
# --- Generate four characteristic yield curve shapes ---
tau = np.linspace(0.25, 30, 200)

# Parameters chosen to produce each shape
shapes = {
    'Normal (upward sloping)': (0.055, -0.020, -0.010, 0.5),
    'Inverted (downward sloping)': (0.030, 0.025, 0.010, 0.5),
    'Humped': (0.050, -0.005, -0.050, 0.8),
    'Flat': (0.045, 0.000, 0.000, 0.5),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_list = [PRIMARY, SECONDARY, TERTIARY, ACCENT]

for ax, (name, params), color in zip(axes.flat, shapes.items(), colors_list):
    rates = nelson_siegel(tau, *params)
    ax.plot(tau, rates * 100, color=color, linewidth=2.5)
    ax.set_xlabel('Maturity (years)')
    ax.set_ylabel('Yield (%)')
    ax.set_title(name)
    ax.set_ylim(2, 7)

plt.suptitle('Characteristic Yield Curve Shapes (Nelson-Siegel)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# All shapes on one plot
fig, ax = plt.subplots(figsize=(10, 6))
for (name, params), color in zip(shapes.items(), colors_list):
    rates = nelson_siegel(tau, *params)
    ax.plot(tau, rates * 100, color=color, linewidth=2, label=name)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.set_title('Yield Curve Shapes Comparison')
ax.legend()
plt.tight_layout()
plt.show()

**Interpreting the output:** Each shape is generated by varying the Nelson-Siegel parameters:
- **Normal:** Negative $\beta_1$ creates the upward slope.
- **Inverted:** Positive $\beta_1$ creates the downward slope.
- **Humped:** A strongly negative $\beta_2$ creates the medium-term hump.
- **Flat:** $\beta_1 = \beta_2 = 0$ leaves only the level component.

This demonstrates the power of the Nelson-Siegel parametrization — just three numbers (plus the decay rate) can produce any commonly observed yield curve shape.> **Key Concept:** These four shapes are not just theoretical curiosities — they occur regularly in real markets. Monitoring the yield curve shape is one of the most important activities for fixed-income portfolio managers, central bank watchers, and macro economists. A change in shape signals a change in economic expectations.

> **CFA Exam Tip:** Be able to identify yield curve shapes from a plot and explain the economic rationale behind each shape. The exam often presents a yield curve and asks you to identify the shape and discuss its implications for economic conditions and investment strategy.


## 10. Summary: Connecting the Pieces

This notebook has taken you through the complete journey of yield curve construction, from raw bond prices to smooth parametric curves. Here is a recap of the key ideas and how they connect:

| Topic | Key Takeaway |
|-------|-------------|
| **Term structure theories** | The curve shape reflects expectations, risk premiums, and supply/demand forces |
| **Bootstrapping** | Extracts spot rates iteratively from coupon bond prices |
| **Forward rates** | The "marginal" rate implied by the spot curve; tradeable via FRAs |
| **Interpolation** | Fills gaps; choice matters most for forward rates |
| **Nelson-Siegel** | 4-parameter model: level + slope + curvature; used by central banks |
| **Svensson** | 6-parameter extension for complex shapes; used by ECB |

> **CFA Exam Tip:** For the exam, focus on: (1) bootstrapping the first 2-3 spot rates by hand, (2) computing forward rates from spot rates, (3) explaining curve shapes using the four term structure theories, and (4) interpreting Nelson-Siegel parameters. These are the most commonly tested topics.

### What Comes Next

In practice, yield curve construction is the *starting point* for fixed-income analysis, not the endpoint. Once you have a curve, you can:

- **Price bonds** by discounting cash flows at the appropriate spot rates
- **Compute duration and convexity** to measure interest rate sensitivity
- **Value swaps and derivatives** using the forward curve
- **Analyze curve trades** (steepeners, flatteners, butterflies) that bet on curve shape changes
- **Estimate the term premium** to separate rate expectations from risk compensation### Formula Reference Card

| Concept | Formula |
|:---|:---|
| Spot rate from par bond (1-yr) | $s_1 = c_1$ (trivially) |
| Bootstrap recursion | $s_n$ from $100 = \sum_{t=1}^{n-1} \frac{c/2}{(1+s_t/2)^{2t}} + \frac{100 + c/2}{(1+s_n/2)^{2n}}$ |
| Forward rate | $f(t_1, t_2) = \left[\frac{(1+s_2)^{t_2}}{(1+s_1)^{t_1}}\right]^{1/(t_2-t_1)} - 1$ |
| Nelson-Siegel | $y(\tau) = \beta_0 + \beta_1[(1-e^{-\lambda\tau})/(\lambda\tau)] + \beta_2[(1-e^{-\lambda\tau})/(\lambda\tau) - e^{-\lambda\tau}]$ |

> **Key Concept:** The yield curve is not just a line on a chart — it's the foundation of all fixed-income analytics. Every bond price, every swap rate, every discount factor can be derived from the spot curve. Master the yield curve and you master fixed income.


## 11. References

1. Nelson, C. R. & Siegel, A. F. "Parsimonious Modeling of Yield Curves." *Journal of Business*, 60(4), 1987.
2. Svensson, L. E. O. "Estimating and Interpreting Forward Interest Rates: Sweden 1992-1994." *IMF Working Paper* 94/114, 1994.
3. Hagan, P. S. & West, G. "Interpolation Methods for Curve Construction." *Applied Mathematical Finance*, 13(2), 2006.
4. Tuckman, B. & Serrat, A. *Fixed Income Securities*, 3rd ed. Wiley, 2011. — Chapters 2-3.
5. BIS. "Zero-Coupon Yield Curves: Technical Documentation." *BIS Papers* No. 25, 2005.
6. CFA Institute. *CFA Program Curriculum Level I and II*, "Fixed Income" volumes.### CFA Level 1 Curriculum Alignment

Key Learning Outcome Statements covered:
- LOS: Describe the relationships among spot rates, forward rates, yield to maturity, and the price of a bond
- LOS: Describe how zero-coupon rates (spot rates) may be obtained from the par curve by bootstrapping
- LOS: Describe the theories of the term structure of interest rates
- LOS: Describe yield curve shapes and their relation to economic conditions
